# Test Hybrid Model on Real Time Series

This notebook trains the existing PyTorch hybrid NN-GARCH model on cleaned WIKI prices returns.
Data batches are sampled via `tf.data.Dataset.from_generator` and then fed into the training functions from `engine.py`.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

from data_stocks_WIKI_price import get_cleaned_data
from model import HybridGarch
from engine import train_hybrid_garch, eval_log_mse
from visualization import (
    plot_training_curves,
    collect_true_pred_sigma2,
    plot_true_pred_scatter,
    plot_predictions_analysis,
)


In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

FILE_PATH = r"C:/Users/karim/OneDrive/Documents/Msc AI/Finance/TP/WIKI_PRICES_212b326a081eacca455e13140d7bb9db.zip"

WINDOW_SIZE = 90
BATCH_SIZE = 500

TRAIN_RANGE = (0, 1000)
VAL_RANGE = (1000, 1200)
TEST_RANGE = (1200, 1400)

TRAIN_STEPS = 150
VAL_STEPS = 40
TEST_STEPS = 40

N_EPOCHS = 12
LR = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
returns_df = get_cleaned_data(FILE_PATH)
print("Cleaned returns shape (time, tickers):", returns_df.shape)

if returns_df.shape[0] < TEST_RANGE[1]:
    raise ValueError(f"Not enough rows for requested split end={TEST_RANGE[1]} with T={returns_df.shape[0]}")

returns_values = returns_df.to_numpy(dtype=np.float32)
print("Date range:", returns_df.index.min(), "->", returns_df.index.max())
print("Number of tickers with zero NaN:", returns_df.shape[1])


In [ ]:
def make_range_generator(values, start_idx, end_idx, window_size=90, batch_size=500, seed=None):
    """Infinite generator over one time split; re-samples indices every batch."""
    n_time, n_tickers = values.shape
    if not (0 <= start_idx < end_idx <= n_time):
        raise ValueError(f"Invalid range [{start_idx}, {end_idx}) for n_time={n_time}")

    min_t = start_idx + window_size
    if min_t >= end_idx:
        raise ValueError("Range too short for the given window_size")

    rng = np.random.default_rng(seed)

    def _generator():
        while True:
            ticker_idx = rng.choice(n_tickers, size=batch_size, replace=(n_tickers < batch_size))
            t_idx = rng.integers(low=min_t, high=end_idx, size=batch_size)

            x = np.empty((batch_size, window_size, 1), dtype=np.float32)
            y = np.empty((batch_size,), dtype=np.float32)

            for i, (t, j) in enumerate(zip(t_idx, ticker_idx)):
                x[i, :, 0] = values[t - window_size:t, j]
                # Target proxy for one-step variance on real data
                y[i] = values[t, j] ** 2

            yield x, y

    return _generator


def make_tf_dataset(generator_fn, batch_size=500, window_size=90):
    output_signature = (
        tf.TensorSpec(shape=(batch_size, window_size, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(batch_size,), dtype=tf.float32),
    )
    return tf.data.Dataset.from_generator(generator_fn, output_signature=output_signature).prefetch(tf.data.AUTOTUNE)


class TorchIterableFromTF:
    """Adapter so engine.py can consume tf.data batches without rewriting training logic."""

    def __init__(self, tf_dataset, steps_per_epoch):
        self.tf_dataset = tf_dataset
        self.steps_per_epoch = int(steps_per_epoch)

    def __iter__(self):
        for xb, yb in self.tf_dataset.take(self.steps_per_epoch).as_numpy_iterator():
            yield torch.from_numpy(xb), torch.from_numpy(yb)

    def __len__(self):
        return self.steps_per_epoch


In [ ]:
train_gen_fn = make_range_generator(returns_values, *TRAIN_RANGE, window_size=WINDOW_SIZE, batch_size=BATCH_SIZE, seed=SEED)
val_gen_fn = make_range_generator(returns_values, *VAL_RANGE, window_size=WINDOW_SIZE, batch_size=BATCH_SIZE, seed=SEED + 1)
test_gen_fn = make_range_generator(returns_values, *TEST_RANGE, window_size=WINDOW_SIZE, batch_size=BATCH_SIZE, seed=SEED + 2)

train_tf = make_tf_dataset(train_gen_fn, batch_size=BATCH_SIZE, window_size=WINDOW_SIZE)
val_tf = make_tf_dataset(val_gen_fn, batch_size=BATCH_SIZE, window_size=WINDOW_SIZE)
test_tf = make_tf_dataset(test_gen_fn, batch_size=BATCH_SIZE, window_size=WINDOW_SIZE)

train_loader = TorchIterableFromTF(train_tf, TRAIN_STEPS)
val_loader = TorchIterableFromTF(val_tf, VAL_STEPS)
test_loader = TorchIterableFromTF(test_tf, TEST_STEPS)

xb, yb = next(iter(train_loader))
print("Feature batch shape:", tuple(xb.shape))
print("Target batch shape:", tuple(yb.shape))


In [ ]:
model = HybridGarch(hidden=32).to(device)

history = train_hybrid_garch(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=LR,
    device=device,
    ckpt_path="hybrid_garch_real.pt",
)


In [ ]:
test_log_mse = eval_log_mse(model, test_loader, device)
print(f"Test log-MSE on split {TEST_RANGE}: {test_log_mse:.6f}")

plot_training_curves(history, test_loss=test_log_mse, title="Hybrid NN-GARCH on WIKI real time series")

y_true, y_pred = collect_true_pred_sigma2(model, test_loader, device, max_batches=TEST_STEPS)
_ = plot_true_pred_scatter(
    y_true,
    y_pred,
    to_volatility=False,
    loglog=True,
    title="Test split (1200-1400): predicted variance proxy vs realized r^2",
)

_ = plot_predictions_analysis(model, test_loader, device, n_examples=1000)
